In [1]:
import numpy as np
import torch
import scipy.stats as stats
from torch.distributions import Uniform
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.inference import simulate_for_sbi
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)
import sys
sys.path.append('../../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim
from discrete_uniform import DiscreteUniform

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [ ]:
torch_device = "cpu"


np.random.seed(100)
tree = ClonalTree(n=15)

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 300

x_o = ClonalOrigin_seq_sim(tree, rho_site, theta_site, L, delta, k_vec=[20, 50, 90])
x_o = torch.tensor(x_o, device=torch_device)
x_o = x_o.flatten()
x_o_numpy = x_o.cpu().numpy()

prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.2]))
prior_delta = Uniform(low=torch.tensor([1.0]), high=torch.tensor([500.0]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.2]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))

prior = MultipleIndependent(
    dists=[prior_rho, prior_delta, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [ ]:
def simulator(theta):
    theta = theta.reshape(-1)
    summary_stats = ClonalOrigin_seq_sim(tree,
                                         theta[0].item(),
                                         theta[2].item(),
                                         int(theta[3].item()),
                                         theta[1].item(),
                                         k_vec=[20, 50, 90])
    summary_stats = torch.tensor(summary_stats, device=torch_device)
    return summary_stats

In [ ]:
theta = prior.sample()
theta

In [ ]:
int(theta[3].item())

In [ ]:
prior, num_parameters, prior_returns_numpy = process_prior(prior)
simulator = process_simulator(simulator, prior, prior_returns_numpy)
check_sbi_inputs(simulator, prior)